# 01 Source Spike and Cluster Check

## Purpose

Verify that the three required data sources and the available FH cluster environment are feasible before implementation continues.

This notebook is a spike notebook. It does not implement the final pipeline, Kafka producer, Spark streaming job, or analysis layer.


## Inputs

- Two pilot cities: Vienna and Berlin.
- Open-Meteo Air Quality API endpoint.
- Wikipedia city pages.
- EEA historical air-quality source documentation and local file expectations.
- FH Spark/Kafka environment notes without secrets.


## Outputs

- Source feasibility matrix.
- Optional tiny Open-Meteo JSON samples under `data/bronze/open_meteo_raw/`.
- Optional tiny Wikipedia HTML samples under `data/bronze/wikipedia_html/`.
- Cluster connectivity and storage decision notes.


## Technologies used

Python, pandas, requests, BeautifulSoup, JSON, Markdown, and optional PySpark smoke-test code.


## Configuration

External source calls are guarded by `RUN_SOURCE_SPIKES`. Set environment variable `RUN_SOURCE_SPIKES=true` to execute the HTTP checks.

Spark cluster code is provided as a smoke-test template only. Parquet-producing notebooks use `SPARK_MASTER_URL=local[*]` unless shared cluster storage is confirmed.


In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))

RUN_SOURCE_SPIKES = os.getenv("RUN_SOURCE_SPIKES", "false").lower() == "true"
OPEN_METEO_BASE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WIKIPEDIA_RAW_DIR = DATA_DIR / "bronze" / "wikipedia_html"
OPEN_METEO_RAW_DIR = DATA_DIR / "bronze" / "open_meteo_raw"
WIKIPEDIA_RAW_DIR.mkdir(parents=True, exist_ok=True)
OPEN_METEO_RAW_DIR.mkdir(parents=True, exist_ok=True)

pilot_cities = pd.DataFrame([
    {"city_id": "vienna_at", "city_name": "Vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de", "city_name": "Berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
])
pilot_cities

## Implementation

### Open-Meteo REST API spike

The REST API check requests PM2.5, PM10 and NO2 using Open-Meteo field names `pm2_5`, `pm10`, and `nitrogen_dioxide`. The spike stores only tiny sample JSON files if execution is enabled.


In [ ]:
def build_open_meteo_params(latitude: float, longitude: float) -> dict:
    return {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "pm2_5,pm10,nitrogen_dioxide",
        "timezone": "UTC",
        "forecast_days": 1,
    }


def fetch_open_meteo_sample(city_row: pd.Series) -> dict:
    import requests

    try:
        response = requests.get(
            OPEN_METEO_BASE_URL,
            params=build_open_meteo_params(city_row["latitude"], city_row["longitude"]),
            timeout=20,
        )
        response.raise_for_status()
        payload = response.json()
        output_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}_sample.json"
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        hourly_keys = sorted(payload.get("hourly", {}).keys())
        return {
            "source": "Open-Meteo",
            "city_id": city_row["city_id"],
            "status": "usable" if {"pm2_5", "pm10", "nitrogen_dioxide"}.issubset(hourly_keys) else "usable with constraints",
            "format": "JSON",
            "relevant_fields": ", ".join(hourly_keys),
            "evidence_path": str(output_path),
            "risks": "Field availability and missing hourly values must be checked during Phase 5.",
        }
    except requests.RequestException as exc:
        return {
            "source": "Open-Meteo",
            "city_id": city_row["city_id"],
            "status": f"spike failed: {exc}",
            "format": "JSON",
            "relevant_fields": "pm2_5, pm10, nitrogen_dioxide",
            "evidence_path": "fetch failed",
            "risks": "Source must be rechecked. Verify network access and API availability.",
        }


open_meteo_results = []
if RUN_SOURCE_SPIKES:
    for _, row in pilot_cities.iterrows():
        open_meteo_results.append(fetch_open_meteo_sample(row))
else:
    open_meteo_results.append({
        "source": "Open-Meteo",
        "city_id": "pilot",
        "status": "not executed in this run",
        "format": "JSON",
        "relevant_fields": "pm2_5, pm10, nitrogen_dioxide",
        "evidence_path": "set RUN_SOURCE_SPIKES=true to create tiny samples",
        "risks": "Source must be rechecked when executing Phase 1.",
    })

pd.DataFrame(open_meteo_results)

### Wikipedia HTML spike

The web scraping spike checks that city HTML can be fetched with a clear User-Agent and stored as raw Bronze evidence.


In [ ]:
def fetch_wikipedia_sample(city_row: pd.Series) -> dict:
    import requests
    from bs4 import BeautifulSoup

    headers = {"User-Agent": "euro-air-quality-pipeline/1.0 educational source spike"}
    try:
        response = requests.get(city_row["wikipedia_url"], headers=headers, timeout=20)
        response.raise_for_status()
        html = response.text
        output_path = WIKIPEDIA_RAW_DIR / f"{city_row['city_id']}_sample.html"
        output_path.write_text(html, encoding="utf-8")
        soup = BeautifulSoup(html, "html.parser")
        return {
            "source": "Wikipedia",
            "city_id": city_row["city_id"],
            "status": "usable" if soup.select_one("table.infobox") else "usable with constraints",
            "format": "HTML",
            "relevant_fields": "infobox, title, possible population/area fields",
            "evidence_path": str(output_path),
            "risks": "HTML structure can change and values may be ambiguous.",
        }
    except requests.RequestException as exc:
        return {
            "source": "Wikipedia",
            "city_id": city_row["city_id"],
            "status": f"spike failed: {exc}",
            "format": "HTML",
            "relevant_fields": "infobox, population, area, coordinates",
            "evidence_path": "fetch failed",
            "risks": "Source must be rechecked. Verify network access.",
        }


wikipedia_results = []
if RUN_SOURCE_SPIKES:
    for _, row in pilot_cities.iterrows():
        wikipedia_results.append(fetch_wikipedia_sample(row))
else:
    wikipedia_results.append({
        "source": "Wikipedia",
        "city_id": "pilot",
        "status": "not executed in this run",
        "format": "HTML",
        "relevant_fields": "infobox, population, area, coordinates",
        "evidence_path": "set RUN_SOURCE_SPIKES=true to create tiny samples",
        "risks": "Parser must be defensive and record parse_status.",
    })

pd.DataFrame(wikipedia_results)

### EEA file/batch source check

EEA is the required file/batch source. The project expects a local CSV or Parquet extract, not an automatic full download in this spike. Phase 3 implements loading, normalization and aggregation.


In [4]:
eea_source_check = pd.DataFrame([
    {
        "source": "EEA historical air quality",
        "type": "file/batch",
        "status": "usable with constraints",
        "format": "CSV or Parquet local extract",
        "relevant_fields": "station, timestamp, pollutant, value, unit",
        "risks": "station-to-city mapping, pollutant coverage, file availability",
        "decision": "Use as historical file/batch source in notebook 03.",
    }
])
eea_source_check


,source,type,status,format,relevant_fields,risks,decision
0,EEA historical air quality,file/batch,usable with constraints,CSV or Parquet local extract,"station, timestamp, pollutant, value, unit","station-to-city mapping, pollutant coverage, f...",Use as historical file/batch source in noteboo...


### FH Spark cluster check

Known finding from prior connectivity tests: Spark master was reachable and a basic DataFrame action worked. HDFS/shared storage was not confirmed, `fs.defaultFS` was observed as `file:///`, and local Jupyter paths are not assumed to be shared with executors.


In [5]:
cluster_findings = pd.DataFrame([
    {"check": "Spark master reachable", "status": "passed", "decision": "cluster can be used for smoke tests"},
    {"check": "Basic DataFrame action", "status": "passed", "decision": "compute connectivity is documented"},
    {"check": "HDFS/shared storage", "status": "not confirmed", "decision": "do not use cluster for project data/ Parquet output"},
    {"check": "Standard pipeline mode", "status": "decided", "decision": "use Spark local[*] for Parquet-producing notebooks"},
])
cluster_findings


,check,status,decision
0,Spark master reachable,passed,cluster can be used for smoke tests
1,Basic DataFrame action,passed,compute connectivity is documented
2,HDFS/shared storage,not confirmed,do not use cluster for project data/ Parquet o...
3,Standard pipeline mode,decided,use Spark local[*] for Parquet-producing noteb...


## Validation / Quality Checks

Validate that all three data sources are represented and that the cluster decision does not overclaim shared storage.


In [ ]:
source_matrix = pd.concat([
    pd.DataFrame(open_meteo_results),
    pd.DataFrame(wikipedia_results),
    eea_source_check.rename(columns={"type": "source_type"}),
], ignore_index=True, sort=False)

expected_sources = {"Open-Meteo", "Wikipedia", "EEA historical air quality"}
missing_sources = expected_sources - set(source_matrix["source"])
assert not missing_sources, f"Source matrix is missing entries for: {missing_sources}"

hdfs_row = cluster_findings.loc[cluster_findings["check"].eq("HDFS/shared storage"), "status"]
assert not hdfs_row.eq("passed").any(), \
    "HDFS/shared storage must not be marked as passed — shared cluster storage is not confirmed for this project"

source_matrix

## Results

Phase 1 establishes that the planned sources are viable or viable with constraints. The actual source executions can be rerun by enabling the guard flag, and generated samples remain ignored by Git.


## Limitations

Source availability can change. EEA data availability depends on local extracts. The FH cluster cannot be used for reliable project-local Parquet output unless shared storage is confirmed.


## Next step

Run notebook `02_city_reference_model.ipynb` to create the central city join model.
